In [ ]:
# !pip install -U ipywidgets jupyter
# python -m pip check
# python -m pip index versions ragas

In [1]:
from ragas import SingleTurnSample, EvaluationDataset
from ragas.metrics import Faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
# from langchain_google_vertexai import VertexAI, VertexAIEmbeddings
from langchain_groq import ChatGroq

# User's question
user_input = "What is the capital of France?"

# Retrieved contexts (e.g., from a knowledge base or search engine)
retrieved_contexts = ["Paris is the capital and most populous city of France."]

# AI's response
response = "The capital of France is Paris."

# Reference answer (ground truth)
reference = "Paris"

# Evaluation rubric
rubric = {
    "accuracy": "Correct",
    "completeness": "High",
    "fluency": "Excellent"
}

# Create the SingleTurnSample instance
# Sample 1
sample1 = SingleTurnSample(
    user_input="What is the capital of Germany?",
    retrieved_contexts=["Berlin is the capital and largest city of Germany."],
    response="The capital of Germany is Berlin.",
    reference="Berlin",
)

# Sample 2
sample2 = SingleTurnSample(
    user_input="Who wrote 'Pride and Prejudice'?",
    retrieved_contexts=["'Pride and Prejudice' is a novel by Jane Austen."],
    response="'Pride and Prejudice' was written by Jane Austen.",
    reference="Jane Austen",
)

# Sample 3
sample3 = SingleTurnSample(
    user_input="What's the chemical formula for water?",
    retrieved_contexts=["Water has the chemical formula H2O."],
    response="The chemical formula for water is H2O.",
    reference="H2O",
)
evaluator_llm =  ChatGroq(
    api_key="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",
    model='openai/gpt-oss-120b'
)


C:\Users\Bhakti Thakur\AppData\Local\Temp\ipykernel_5884\1142325704.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness


In [ ]:
dataset = EvaluationDataset(samples=[sample1, sample2, sample3])

In [3]:
# dataset = EvaluationDataset(samples=[sample])
# scorer = Faithfulness(llm=evaluator_llm)
# await scorer.single_turn_ascore(dataset)

In [1]:
import os
import asyncio
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI     # async client — required by ragas abatch_score

In [2]:
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings
from ragas import SingleTurnSample
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    AnswerCorrectness,
)

# ── DeepEval ──────────────────────────────────────────────────────────────────
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import ToolCorrectnessMetric


load_dotenv()
print("✅ All imports loaded")

✅ All imports loaded


In [4]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
JUDGE_GROQ_API_KEY = os.getenv("JUDGE_GROQ_API", GROQ_API_KEY)

groq_client = AsyncOpenAI(
    api_key=JUDGE_GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

judge_llm = llm_factory("openai/gpt-oss-120b", provider="openai", client=groq_client)

# ── Local Embeddings (zero API cost) ──────────────────────────────────────────
ragas_embeddings = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2", #384
    use_api=False,
)

print(f"✅ Judge LLM  : {type(judge_llm).__name__}")
print(f"✅ Embeddings : {type(ragas_embeddings).__name__} (local — no API key needed)")
using = "JUDGE_GROQ" if os.getenv("JUDGE_GROQ") else "GROQ_API_KEY (fallback)"
print(f"✅ Using key  : {using}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Judge LLM  : InstructorLLM
✅ Embeddings : HuggingFaceEmbeddings (local — no API key needed)
✅ Using key  : GROQ_API_KEY (fallback)


In [5]:
async def async_cooldown(seconds=60):
    """Async cooldown — works inside Jupyter's running event loop."""
    print(f"\n⏳ Cooldown {seconds}s (Groq rate-limit buffer)...", end=" ")
    for _ in range(seconds // 10):
        await asyncio.sleep(10)
        print(".", end="", flush=True)
    print("  ✅ Ready.\n")
    
    
# scores --  0 - 1

def show_scores(df, col, title):
    """Colour-coded score table for one metric column."""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    for i, row in df.iterrows():
        score = row.get(col, float("nan"))
        if score != score:
            bar = "⬜"
        elif score >= 0.75:
            bar = "🟢"
        elif score >= 0.5:
            bar = "🟡"
        else:
            bar = "🔴"
        q = str(row.get("user_input", f"Sample {i+1}"))[:55]
        print(f"  {bar}  {score:.2f}  |  {q}")
    avg = df[col].mean()
    label = "✅ Good" if avg >= 0.75 else ("⚠️  Fair" if avg >= 0.5 else "❌ Poor")
    print(f"{'─'*60}")
    print(f"  AVG: {avg:.2f}  {label}")
    print(f"{'='*60}\n")

In [7]:
hall_samples = [
    # Good: answer is grounded in context only
    SingleTurnSample(
        user_input="Can I return a used laptop?",
        retrieved_contexts=[
            "Our return policy allows returns within 30 days of purchase for unused, unopened electronics.",
            "Items must be in original packaging with all accessories included.",
        ],
        response="According to the return policy, laptops must be unused and in original packaging to be eligible for a return within 30 days.",
        reference="No. Only unused, unopened electronics in original packaging qualify for returns within 30 days.",
    ),
    # Medium: response adds a fabricated 10% coupon not in any context chunk
    SingleTurnSample(
        user_input="What is the refund timeline?",
        retrieved_contexts=[
            "Approved refunds are processed within 5 to 7 business days.",
        ],
        response="Refunds are processed in 5 to 7 business days and you will also receive a 10%\ discount coupon on your next order.",
        reference="Refunds are processed within 5 to 7 business days.",
    ),
    # Bad: complete contradiction — context says customer pays, answer says free
    SingleTurnSample(
        user_input="Do you offer free return shipping?",
        retrieved_contexts=[
            "Customers are responsible for return shipping costs unless the item is defective.",
        ],
        response="Yes, we provide free prepaid return shipping labels for all returns including buyer remorse returns.",
        reference="No. Customers pay return shipping unless the item is defective.",
    ),
]

print("Running Exp 1 — Faithfulness (E-commerce Returns)...")
_m = Faithfulness(llm=judge_llm)
_inputs = [
    {"user_input": s.user_input, "response": s.response, "retrieved_contexts": s.retrieved_contexts}
    for s in hall_samples
]
_res = await _m.abatch_score(_inputs)
df_hall = pd.DataFrame([
    {"user_input": s.user_input, "faithfulness": float(r.value)}
    for s, r in zip(hall_samples, _res)
])
show_scores(df_hall, "faithfulness", "Exp 1 — Faithfulness (E-commerce Returns)")
await async_cooldown(60)

Running Exp 1 — Faithfulness (E-commerce Returns)...

  Exp 1 — Faithfulness (E-commerce Returns)
  🟢  1.00  |  Can I return a used laptop?
  🟡  0.50  |  What is the refund timeline?
  🔴  0.00  |  Do you offer free return shipping?
────────────────────────────────────────────────────────────
  AVG: 0.50  ⚠️  Fair


⏳ Cooldown 60s (Groq rate-limit buffer)... ......  ✅ Ready.



In [8]:
rel_samples = [
    # Good: directly and specifically answers the visa question
    SingleTurnSample(
        user_input="What are the visa requirements for visiting Japan from India?",
        retrieved_contexts=[
            "Indian passport holders require a tourist visa to enter Japan.",
            "Visa applications must be submitted at the Japanese embassy with a valid passport, photos, and a travel itinerary.",
        ],
        response="Indian citizens need a tourist visa to enter Japan. You apply at the Japanese embassy with your passport, recent photos, and a travel itinerary.",
        reference="Indian passport holders need a tourist visa for Japan. Apply at the Japanese embassy with passport, photos, and itinerary.",
    ),
    # Medium: vague — fails to mention any specific places
    SingleTurnSample(
        user_input="What are the best places to visit in Japan in April?",
        retrieved_contexts=[
            "April is cherry blossom season in Japan. Popular spots include Ueno Park in Tokyo, Maruyama Park in Kyoto, and Hirosaki Castle in Aomori.",
        ],
        response="Japan is a wonderful country with many beautiful places to visit. There are so many things to see and do throughout the year.",
        reference="In April, top cherry blossom spots are Ueno Park (Tokyo), Maruyama Park (Kyoto), and Hirosaki Castle (Aomori).",
    ),
    # Bad: off-topic — asked about buying a rail pass, answered about food
    SingleTurnSample(
        user_input="How do I buy a Japan Rail Pass?",
        retrieved_contexts=[
            "The Japan Rail Pass can be purchased online before traveling or at major JR stations on arrival.",
            "It covers unlimited travel on most JR trains including the Shinkansen bullet train.",
        ],
        response="Japanese cuisine is incredible. Must-try dishes include sushi, ramen, and tempura during your visit.",
        reference="You can buy a Japan Rail Pass online before your trip or at major JR stations in Japan.",
    ),
]


print("Running Exp 2 — Answer Relevancy (Travel Planning)...")
_m = AnswerRelevancy(llm=judge_llm, embeddings=ragas_embeddings)
_inputs = [
    {"user_input": s.user_input, "response": s.response}
    for s in rel_samples
]
_res = await _m.abatch_score(_inputs)
df_rel = pd.DataFrame([
    {"user_input": s.user_input, "answer_relevancy": float(r.value)}
    for s, r in zip(rel_samples, _res)
])
show_scores(df_rel, "answer_relevancy", "Exp 2 — Answer Relevancy (Travel Planning)")
await async_cooldown(60)

Running Exp 2 — Answer Relevancy (Travel Planning)...

  Exp 2 — Answer Relevancy (Travel Planning)
  🟢  0.85  |  What are the visa requirements for visiting Japan from 
  🔴  0.42  |  What are the best places to visit in Japan in April?
  🔴  0.30  |  How do I buy a Japan Rail Pass?
────────────────────────────────────────────────────────────
  AVG: 0.52  ⚠️  Fair


⏳ Cooldown 60s (Groq rate-limit buffer)... ......  ✅ Ready.



In [11]:
# ── Scenario: Python Coding Help ─────────────────────────────────────────────
prec_samples = [
    # Good: relevant chunks first, noise last
    SingleTurnSample(
        user_input="How do I read a CSV file in Python?",
        retrieved_contexts=[
            "Use pandas: import pandas as pd; df = pd.read_csv('file.csv') to load a CSV into a DataFrame.",
            "pd.read_csv() accepts parameters like sep, header, encoding, and index_col for customisation.",
            "Python lists are ordered, mutable sequences defined with square brackets.",
        ],
        response="To read a CSV in Python, use pandas: import pandas as pd; df = pd.read_csv('file.csv'). You can also set sep, header, and encoding.",
        reference="Use pandas pd.read_csv('file.csv') to read a CSV file into a DataFrame.",
    ),
    # Medium: one irrelevant chunk before the useful one
    SingleTurnSample(
        user_input="How do I reverse a string in Python?",
        retrieved_contexts=[
            "Python supports various data types: int, float, string, list, and dict.",
            "To reverse a string, use slicing: s[::-1] or use ''.join(reversed(s)).",
        ],
        response="You can reverse a string using slicing: s[::-1].",
        reference="Reverse a string with s[::-1] or ''.join(reversed(s)).",
    ),
    # Bad: two irrelevant chunks before the single useful chunk
    SingleTurnSample(
        user_input="How do I sort a list in Python?",
        retrieved_contexts=[
            "Python decorators wrap functions to add behaviour using the @ symbol.",
            "The Global Interpreter Lock (GIL) affects multithreaded Python performance.",
            "Use list.sort() for in-place sorting or sorted(list) for a new sorted list.",
        ],
        response="Use list.sort() to sort in-place or sorted(list) to get a new sorted list.",
        reference="Sort a list with list.sort() (in-place) or sorted(list) (returns new list).",
    ),
]

print("Running Exp 3 — Context Precision (Python Coding)...")
_m = ContextPrecision(llm=judge_llm)
_inputs = [
    {"user_input": s.user_input, "reference": s.reference, "retrieved_contexts": s.retrieved_contexts}
    for s in prec_samples
]
_res = await _m.abatch_score(_inputs)
df_prec = pd.DataFrame([
    {"user_input": s.user_input, "context_precision": float(r.value)}
    for s, r in zip(prec_samples, _res)
])
df_prec

# show_scores(df_prec, "context_precision", "Exp 3 — Context Precision (Python Coding)")
# await async_cooldown(60)

Running Exp 3 — Context Precision (Python Coding)...


,user_input,context_precision
0,How do I read a CSV file in Python?,1.000000
1,How do I reverse a string in Python?,0.500000
2,How do I sort a list in Python?,0.333333


In [13]:
# ── Scenario: Cooking / Recipes ──────────────────────────────────────────────
recall_samples = [
    # Good: all 4 steps present in context
    SingleTurnSample(
        user_input="How do I make a classic French omelette?",
        retrieved_contexts=[
            "Crack 3 eggs into a bowl and whisk until smooth. Season with salt and pepper.",
            "Heat butter in a non-stick pan over medium heat until foamy but not brown.",
            "Pour the eggs in and stir with a spatula while shaking the pan for 30 seconds.",
            "Fold the omelette into thirds and slide onto a plate — it should be pale yellow, not browned.",
        ],
        response="Whisk 3 eggs with salt and pepper. Melt butter until foamy, pour eggs in, stir and shake for 30 seconds, fold into thirds. Serve pale yellow.",
        reference="Whisk 3 eggs, season, cook in foamy butter over medium heat stirring constantly, fold into thirds. The omelette should stay pale, not browned.",
    ),
    # Medium: missing oven temp and bake time for banana bread
    SingleTurnSample(
        user_input="What are the steps to make banana bread?",
        retrieved_contexts=[
            "Mash 3 ripe bananas in a mixing bowl.",
            "Mix in 75g melted butter, 150g sugar, 1 egg, and 1 tsp vanilla extract.",
        ],
        response="Mash the bananas, mix with butter, sugar, egg, and vanilla, pour into a loaf tin and bake until golden.",
        reference="Mash 3 bananas, mix with butter, sugar, egg, vanilla, fold in 190g flour with baking soda and salt. Bake at 175 C for 60 minutes.",
    ),
    # Bad: context has nothing relevant to cooking pasta
    SingleTurnSample(
        user_input="How long should I cook spaghetti?",
        retrieved_contexts=[
            "Pasta was introduced to Italy from Arab traders in the 12th century.",
            "There are over 350 recognised pasta shapes in Italian culinary tradition.",
        ],
        response="Cook pasta according to the package instructions until al dente.",
        reference="Boil spaghetti in salted water for 8 to 10 minutes. Taste one strand 1 to 2 minutes before the package time for al dente texture.",
    ),
]


print("Running Exp 4 — Context Recall (Cooking / Recipes)...")
_m = ContextRecall(llm=judge_llm)
_inputs = [
    {"user_input": s.user_input, "retrieved_contexts": s.retrieved_contexts, "reference": s.reference}
    for s in recall_samples
]
_res = await _m.abatch_score(_inputs)
df_recall = pd.DataFrame([
    {"user_input": s.user_input, "context_recall": float(r.value)}
    for s, r in zip(recall_samples, _res)
])

# df_recall
show_scores(df_recall, "context_recall", "Exp 4 — Context Recall (Cooking / Recipes)")
await async_cooldown(60)

Running Exp 4 — Context Recall (Cooking / Recipes)...

  Exp 4 — Context Recall (Cooking / Recipes)
  🟢  1.00  |  How do I make a classic French omelette?
  🔴  0.00  |  What are the steps to make banana bread?
  🔴  0.00  |  How long should I cook spaghetti?
────────────────────────────────────────────────────────────
  AVG: 0.33  ❌ Poor


⏳ Cooldown 60s (Groq rate-limit buffer)... ......  ✅ Ready.



In [12]:
correct_samples = [
    # Good: fully correct, matches reference
    SingleTurnSample(
        user_input="Who invented the telephone?",
        retrieved_contexts=[
            "Alexander Graham Bell is credited with patenting the first practical telephone in 1876.",
            "Bell demonstrated it by speaking to his assistant Thomas Watson in the next room.",
        ],
        response="Alexander Graham Bell invented the telephone and received the patent in 1876.",
        reference="Alexander Graham Bell invented the telephone, patenting it in 1876.",
    ),
    # Medium: right topic, wrong number
    SingleTurnSample(
        user_input="What is the boiling point of water at sea level?",
        retrieved_contexts=[
            "Water boils at 100 degrees Celsius (212 F) at sea level under standard atmospheric pressure.",
        ],
        response="Water boils at 90 degrees Celsius at sea level.",
        reference="Water boils at 100 degrees Celsius (212 F) at sea level.",
    ),
    # Bad: completely wrong answer
    SingleTurnSample(
        user_input="Which planet is closest to the Sun?",
        retrieved_contexts=[
            "Mercury is the closest planet to the Sun, orbiting at an average distance of 57.9 million km.",
        ],
        response="Venus is the closest planet to the Sun.",
        reference="Mercury is the closest planet to the Sun.",
    ),
]

print("Running Exp 5 — Answer Correctness (General Knowledge)...")
_m = AnswerCorrectness(llm=judge_llm, embeddings=ragas_embeddings)
_inputs = [
    {"user_input": s.user_input, "response": s.response, "reference": s.reference}
    for s in correct_samples
]
_res = await _m.abatch_score(_inputs)
df_correct = pd.DataFrame([
    {"user_input": s.user_input, "answer_correctness": float(r.value)}
    for s, r in zip(correct_samples, _res)
])
show_scores(df_correct, "answer_correctness", "Exp 5 — Answer Correctness (General Knowledge)")
await async_cooldown(60)

Running Exp 5 — Answer Correctness (General Knowledge)...

  Exp 5 — Answer Correctness (General Knowledge)
  🟢  1.00  |  Who invented the telephone?
  🔴  0.24  |  What is the boiling point of water at sea level?
  🔴  0.21  |  Which planet is closest to the Sun?
────────────────────────────────────────────────────────────
  AVG: 0.48  ❌ Poor


⏳ Cooldown 60s (Groq rate-limit buffer)... ......  ✅ Ready.



## GATEWAYS